# Многомерное исчисление для ML

## Модуль 5: Частные производные, градиенты, якобианы

В этом ноутбуке мы рассмотрим основы многомерного исчисления, необходимые для понимания машинного обучения.

### Содержание:
1. Частные производные
2. Градиент
3. Якобиан
4. Гессиан
5. Правило цепочки (chain rule)
6. Градиентный спуск

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Частные производные

**Частная производная** — это производная функции нескольких переменных по одной из них, при фиксированных остальных.

### Определение:
Для функции $f(x, y)$:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x, y)}{h}$$

$$\frac{\partial f}{\partial y} = \lim_{h \to 0} \frac{f(x, y+h) - f(x, y)}{h}$$

### Примеры:
1. $f(x, y) = x^2 + y^2$
   - $\frac{\partial f}{\partial x} = 2x$
   - $\frac{\partial f}{\partial y} = 2y$

2. $f(x, y) = xy + \sin(x)$
   - $\frac{\partial f}{\partial x} = y + \cos(x)$
   - $\frac{\partial f}{\partial y} = x$

In [ ]:
# Пример: Частные производные

# Функция
def f(x, y):
    return x**2 + y**2

# Аналитические частные производные
def df_dx(x, y):
    return 2*x

def df_dy(x, y):
    return 2*y

# Численная проверка
def partial_derivative_numerical(f, x, y, var='x', h=1e-6):
    """Численная частная производная"""
    if var == 'x':
        return (f(x + h, y) - f(x - h, y)) / (2 * h)
    else:
        return (f(x, y + h) - f(x, y - h)) / (2 * h)

# Проверка
x, y = 2, 3
print('Частные производные f(x,y) = x² + y²')
print('=' * 60)
print(f'Точка: ({x}, {y})')
print(f'\n∂f/∂x аналитическая: {df_dx(x, y)}')
print(f'∂f/∂x численная: {partial_derivative_numerical(f, x, y, "x"):.10f}')
print(f'\n∂f/∂y аналитическая: {df_dy(x, y)}')
print(f'∂f/∂y численная: {partial_derivative_numerical(f, x, y, "y"):.10f}')

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x_range = np.linspace(-3, 3, 100)
y_range = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = f(X, Y)

# Функция
im = axes[0].contour(X, Y, Z, levels=20, cmap='viridis')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('f(x,y) = x² + y²')
plt.colorbar(im, ax=axes[0])

# ∂f/∂x
Z_dx = df_dx(X, Y)
im = axes[1].contour(X, Y, Z_dx, levels=20, cmap='coolwarm')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('∂f/∂x = 2x')
plt.colorbar(im, ax=axes[1])

# ∂f/∂y
Z_dy = df_dy(X, Y)
im = axes[2].contour(X, Y, Z_dy, levels=20, cmap='coolwarm')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].set_title('∂f/∂y = 2y')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## 2. Градиент

**Градиент** — это вектор частных производных функции.

$$\nabla f = \begin{pmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{pmatrix}$$

### Свойства:
1. Градиент указывает направление наискорейшего возрастания функции
2. $-\nabla f$ указывает направление наискорейшего убывания
3. Градиент перпендикулярен линиям уровня

### Применение в ML:
**Градиентный спуск:**
$$\theta_{t+1} = \theta_t - \alpha \nabla J(\theta_t)$$

где $\alpha$ — скорость обучения (learning rate).

In [ ]:
# Пример: Градиент и градиентный спуск

# Простая функция для демонстрации
def simple_func(x, y):
    return x**2 + 2*y**2

def simple_grad(x, y):
    return np.array([2*x, 4*y])

# Градиентный спуск
def gradient_descent(grad_func, start, learning_rate=0.1, n_steps=50):
    """Градиентный спуск"""
    path = [start]
    current = np.array(start, dtype=float)
    
    for _ in range(n_steps):
        grad = grad_func(current[0], current[1])
        current = current - learning_rate * grad
        path.append(current.copy())
    
    return np.array(path)

# Запуск градиентного спуска
start = np.array([2, 2])
path = gradient_descent(simple_grad, start, learning_rate=0.1, n_steps=20)

print('Градиентный спуск')
print('=' * 60)
print(f'Начальная точка: {start}')
print(f'Конечная точка: {path[-1]}')
print(f'Значение функции: {simple_func(path[-1][0], path[-1][1]):.6f}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Линии уровня и путь
x_range = np.linspace(-3, 3, 100)
y_range = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = simple_func(X, Y)

axes[0].contour(X, Y, Z, levels=20, cmap='viridis')
axes[0].plot(path[:, 0], path[:, 1], 'ro-', linewidth=2, markersize=5)
axes[0].plot(path[0, 0], path[0, 1], 'go', markersize=10, label='Начало')
axes[0].plot(path[-1, 0], path[-1, 1], 'bo', markersize=10, label='Конец')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Градиентный спуск на линиях уровня')
axes[0].legend()

# 3D поверхность
ax = fig.add_subplot(122, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5)
ax.plot(path[:, 0], path[:, 1], 
        [simple_func(p[0], p[1]) for p in path], 
        'ro-', linewidth=2, markersize=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')
ax.set_title('3D визуализация')

plt.tight_layout()
plt.show()

## 3. Якобиан

**Якобиан** — это матрица частных производных вектор-функции.

Для функции $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$:

$$J = \begin{pmatrix} \frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \cdots & \frac{\partial f_2}{\partial x_n} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \cdots & \frac{\partial f_m}{\partial x_n} \end{pmatrix}$$

### Применение:
- Преобразование переменных
- Изменение объёма при преобразовании
- Обратное распространение ошибки (backpropagation)

In [ ]:
# Пример: Якобиан

# Вектор-функция: f(x, y) = (x² + y, xy)
def f_vec(x, y):
    return np.array([x**2 + y, x*y])

# Якобиан (аналитический)
def jacobian_analytical(x, y):
    return np.array([
        [2*x, 1],    # ∂f1/∂x, ∂f1/∂y
        [y, x]       # ∂f2/∂x, ∂f2/∂y
    ])

# Якобиан (численный)
def jacobian_numerical(f_vec, x, y, h=1e-6):
    """Численный якобиан"""
    f0 = f_vec(x, y)
    n = len(f0)
    J = np.zeros((n, 2))
    
    # ∂f/∂x
    J[:, 0] = (f_vec(x + h, y) - f_vec(x - h, y)) / (2 * h)
    # ∂f/∂y
    J[:, 1] = (f_vec(x, y + h) - f_vec(x, y - h)) / (2 * h)
    
    return J

# Проверка
x, y = 2, 3
print('Якобиан для f(x,y) = (x²+y, xy)')
print('=' * 60)
print(f'Точка: ({x}, {y})')
print(f'\nАналитический якобиан:')
print(jacobian_analytical(x, y))
print(f'\nЧисленный якобиан:')
print(jacobian_numerical(f_vec, x, y))

# Визуализация якобиана
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_range = np.linspace(-3, 3, 20)
y_range = np.linspace(-3, 3, 20)
X, Y = np.meshgrid(x_range, y_range)

# Вычисление якобиана в каждой точке
det_J = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        J = jacobian_analytical(X[i, j], Y[i, j])
        det_J[i, j] = np.linalg.det(J)

# Определитель якобиана
im = axes[0].contourf(X, Y, det_J, levels=20, cmap='coolwarm')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Определитель якобиана')
plt.colorbar(im, ax=axes[0])

# Векторное поле градиента
U = 2 * X  # ∂f1/∂x
V = Y      # ∂f2/∂y
axes[1].quiver(X, Y, U, V, alpha=0.5)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Векторное поле градиента')

plt.tight_layout()
plt.show()

## 4. Гессиан

**Гессиан** — это матрица вторых частных производных.

$$H = \begin{pmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} & \cdots \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix}$$

### Применение:
- Определение типа критической точки (минимум, максимум, седловая точка)
- Метод Ньютона для оптимизации
- Анализ кривизны функции

In [ ]:
# Пример: Гессиан

# Функция
def f(x, y):
    return x**2 + 2*y**2 + x*y

# Гессиан (аналитический)
def hessian_analytical(x, y):
    return np.array([
        [2, 1],    # ∂²f/∂x², ∂²f/∂x∂y
        [1, 4]     # ∂²f/∂y∂x, ∂²f/∂y²
    ])

# Проверка
x, y = 1, 2
print('Гессиан для f(x,y) = x² + 2y² + xy')
print('=' * 60)
print(f'Точка: ({x}, {y})')
print(f'\nГессиан:')
H = hessian_analytical(x, y)
print(H)
print(f'\nСобственные значения: {np.linalg.eigvals(H)}')
print(f'Определитель: {np.linalg.det(H)}')
print(f'\nТип точки: Минимум (H положительно определён)')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_range = np.linspace(-3, 3, 100)
y_range = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = f(X, Y)

# Линии уровня
axes[0].contour(X, Y, Z, levels=20, cmap='viridis')
axes[0].plot(x, y, 'ro', markersize=10)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Линии уровня f(x,y)')

# 3D поверхность
ax = fig.add_subplot(122, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5)
ax.scatter([x], [y], [f(x, y)], color='red', s=100, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')
ax.set_title('3D поверхность')

plt.tight_layout()
plt.show()

## 5. Правило цепочки (Chain Rule)

### Для одной переменной:
$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

### Для нескольких переменных:
Если $z = f(u, v)$, где $u = g(x, y)$ и $v = h(x, y)$:

$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial u} \cdot \frac{\partial u}{\partial x} + \frac{\partial z}{\partial v} \cdot \frac{\partial v}{\partial x}$$

### Применение в нейронных сетях:
**Обратное распространение ошибки (backpropagation)** — это последовательное применение правила цепочки для вычисления градиентов.

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w}$$

In [ ]:
# Пример: Правило цепочки

# Простая нейронная сеть: y = σ(w*x + b)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Функция потерь: MSE
def mse_loss(y_true, y_pred):
    return (y_true - y_pred)**2

def mse_loss_derivative(y_true, y_pred):
    return -2 * (y_true - y_pred)

# Прямое распространение
def forward(x, w, b):
    z = w * x + b
    a = sigmoid(z)
    return z, a

# Обратное распространение (цепочка)
def backward(x, w, b, y_true):
    # Прямое распространение
    z, y_pred = forward(x, w, b)
    
    # Обратное распространение (правило цепочки)
    dL_dy = mse_loss_derivative(y_true, y_pred)
    dy_dz = sigmoid_derivative(z)
    dz_dw = x
    dz_db = 1
    
    # Градиенты
    dL_dw = dL_dy * dy_dz * dz_dw
    dL_db = dL_dy * dy_dz * dz_db
    
    return dL_dw, dL_db, y_pred

# Пример
x = 2.0
w = 0.5
b = 0.1
y_true = 1.0

z, y_pred = forward(x, w, b)
dL_dw, dL_db, _ = backward(x, w, b, y_true)

print('Правило цепочки для нейронной сети')
print('=' * 60)
print(f'Вход: x = {x}')
print(f'Параметры: w = {w}, b = {b}')
print(f'Истинное значение: y = {y_true}')
print(f'\nПрямое распространение:')
print(f'  z = w*x + b = {z}')
print(f'  y_pred = σ(z) = {y_pred:.4f}')
print(f'  Loss = (y - y_pred)² = {mse_loss(y_true, y_pred):.4f}')
print(f'\nОбратное распространение (цепочка):')
print(f'  ∂L/∂y = {mse_loss_derivative(y_true, y_pred):.4f}')
print(f'  ∂y/∂z = σ\'(z) = {sigmoid_derivative(z):.4f}')
print(f'  ∂z/∂w = x = {x}')
print(f'  ∂z/∂b = 1')
print(f'\n  ∂L/∂w = ∂L/∂y · ∂y/∂z · ∂z/∂w = {dL_dw:.4f}')
print(f'  ∂L/∂b = ∂L/∂y · ∂y/∂z · ∂z/∂b = {dL_db:.4f}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Сигмоида
z_range = np.linspace(-5, 5, 100)
axes[0].plot(z_range, sigmoid(z_range), 'b-', linewidth=2, label='σ(z)')
axes[0].plot(z_range, sigmoid_derivative(z_range), 'r-', linewidth=2, label="σ'(z)")
axes[0].axvline(z, color='green', linestyle='--', linewidth=2, label=f'z = {z:.2f}')
axes[0].set_xlabel('z')
axes[0].set_ylabel('Значение')
axes[0].set_title('Сигмоида и её производная')
axes[0].legend()

# Компутационный граф (упрощённый)
axes[1].text(0.1, 0.8, f'x = {x}', fontsize=12, ha='center')
axes[1].text(0.3, 0.8, f'w = {w}', fontsize=12, ha='center')
axes[1].text(0.5, 0.8, f'b = {b}', fontsize=12, ha='center')
axes[1].text(0.3, 0.6, f'z = w*x + b = {z:.2f}', fontsize=12, ha='center')
axes[1].text(0.3, 0.4, f'y = σ(z) = {y_pred:.4f}', fontsize=12, ha='center')
axes[1].text(0.3, 0.2, f'L = (y-ŷ)² = {mse_loss(y_true, y_pred):.4f}', fontsize=12, ha='center')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].set_title('Компутационный граф')

plt.tight_layout()
plt.show()

## Упражнения

### Упражнение 1: Частные производные
Вычислите частные производные функции $f(x, y) = x^3 y^2 + \sin(xy)$:
1. $\frac{\partial f}{\partial x}$
2. $\frac{\partial f}{\partial y}$
3. Проверьте численно в точке (1, 2)

### Упражнение 2: Градиент
Для функции $f(x, y) = e^{-(x^2 + y^2)}$:
1. Вычислите градиент
2. Найдите точку максимума
3. Визуализируйте градиентное поле

### Упражнение 3: Якобиан
Вычислите якобиан для функции:
$$\mathbf{f}(x, y) = \begin{pmatrix} x^2 + y \\ xy \\ \sin(x) \end{pmatrix}$$

### Упражнение 4: Правило цепочки
Для функции $z = x^2 + y^2$, где $x = \sin(t)$ и $y = \cos(t)$:
1. Вычислите $\frac{dz}{dt}$ используя правило цепочки
2. Проверьте, подставив $x$ и $y$ в $z$ и вычислив производную напрямую

---

**Решения** можно найти в ноутбуке `solutions/13_Solutions.ipynb`